# REPEAT: Improving Uncertainty Estimation in Representation Learning Explainability

This notebook illustrates the basic usage of REPEAT, a framework for representation learning explainability with improved uncertainty estimation. If you are running this notebook on Google Colab, remember to enable GPU support to speed up computation.

REPEAT treats each pixel in an image as a Bernoulli random variable that is either important or unimportant to the representation of the image. From these Bernoulli random variables we can directly estimate the probability of a pixel being important, and the associated uncertainty, thus enabling users to ascertain certainty in pixel importance. For more information see the [arXiv paper](https://arxiv.org/abs/2412.08513) or the [AAAI 2025 paper](https://doi.org/10.1609/aaai.v39i8.32900).

The notebook consists of the following steps:

1. Install and import packages: installs and imports the necessary packages.
2. Download example image: downloads an example image of a bird.
3. Function for rescaling and displaying images: a helper function for plotting.
4. Load and prepare example image: loads the example image, transforms it into a suitable format, and plots it.
5. Load feature extractor: defines a function that loads a pretrained ResNet18 feature extractor.
6. Run REPEAT: runs REPEAT on the example image.
7. Display results: shows the probability of importance and the associated uncertainty.


In [ ]:
# @title Install and import packages

!pip install repeat-xai

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from relax_xai.utils import imagenet_image_transforms
from torchvision.transforms.functional import pil_to_tensor

from repeat_xai import REPEAT

In [ ]:
# @title Download example image
# @markdown We download an image of a bird from https://commons.wikimedia.org
# @markdown to illustrate the usage of REPEAT.

!wget 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/ae/Tringa_totanus-pjt.jpg/640px-Tringa_totanus-pjt.jpg'

In [ ]:
# @title Function for rescaling and displaying images.
# @markdown This function is taken from the TorchRay library
# @markdown (https://github.com/facebookresearch/TorchRay).


def imsc(img, *args, quiet=False, lim=None, interpolation="lanczos", **kwargs):
    if isinstance(img, Image.Image):
        img = pil_to_tensor(img)
    with torch.no_grad():
        if not lim:
            lim = [img.min(), img.max()]
        img = img - lim[0]  # also makes a copy
        img.mul_(1 / (lim[1] - lim[0]))
        img = torch.clamp(img, min=0, max=1)
        if not quiet:
            bitmap = img.expand(3, *img.shape[1:]).permute(1, 2, 0).cpu().numpy()
    return bitmap

In [ ]:
# @title Load and prepare example image
# @markdown This cell loads the image we downloaded earlier and prepares it by
# @markdown reshaping and normalizing it into a suitable input format. We also
# @markdown plot the image to show what our bird looks like.

example_image = pil_to_tensor(Image.open("/content/640px-Tringa_totanus-pjt.jpg")).float()
example_image = imagenet_image_transforms(device="cuda", new_shape_of_image=224)(example_image)

plt.figure(1, figsize=(5, 5))
plt.imshow(imsc(example_image.squeeze()))
plt.axis("off")
plt.show()

In [ ]:
# @title Load feature extractor
# @markdown This cell creates a function for loading the feature extractor
# @markdown considered in this notebook. An image is represented by the output
# @markdown of the final average pooling layer of a ResNet18 pretrained on
# @markdown Imagenet.


def load_resnet18_encoder() -> nn.Module:
    resnet18 = torchvision.models.resnet18(weights="DEFAULT")
    modules = list(resnet18.children())[:-1]
    encoder = nn.Sequential(*modules, nn.Flatten())
    encoder.eval()
    return encoder

In [ ]:
# @title Run REPEAT on the example image
# @markdown REPEAT estimates the probability of each pixel being important
# @markdown for the representation of the image, by repeatedly running RELAX
# @markdown and thresholding each importance map into important and
# @markdown unimportant pixels. The encoder is automatically moved to the
# @markdown device of the input image. Masking-related hyperparameters (e.g.
# @markdown `num_cells` and `probability_of_drop`) can be passed to
# @markdown `repeat.forward(...)`.

encoder = load_resnet18_encoder()

repeat = REPEAT(example_image, encoder)
repeat.forward()

In [ ]:
# @title Display results
# @markdown The probability of importance shows which pixels are important
# @markdown for the representation of this image for the ResNet18 encoder, and
# @markdown the uncertainty shows how certain we are of the pixel importance.
# @markdown Pixels that are consistently important or unimportant across the
# @markdown repeated explanations have low uncertainty, while pixels that
# @markdown fluctuate between the two have high uncertainty.

fig = plt.figure(figsize=(14, 4))

ax1 = fig.add_subplot(1, 3, 1)
ax1.imshow(imsc(example_image.squeeze()))
ax1.axis("off")
ax1.set_title("input image")

ax2 = fig.add_subplot(1, 3, 2)
ax2.imshow(imsc(example_image.squeeze()))
im2 = ax2.imshow(repeat.probability_of_importance.numpy(force=True), alpha=0.75, cmap="bwr")
ax2.axis("off")
ax2.set_title("probability of importance")
plt.colorbar(im2, fraction=0.046, pad=0.04)

ax3 = fig.add_subplot(1, 3, 3)
ax3.imshow(imsc(example_image.squeeze()))
im3 = ax3.imshow(repeat.uncertainty.numpy(force=True), alpha=0.75, cmap="bwr")
ax3.axis("off")
ax3.set_title("uncertainty")
plt.colorbar(im3, fraction=0.046, pad=0.04)

plt.show()